# Humpback Whale Identification — Contrastive Loss Metric Learning

**Course**: Machine Learning for Computer Vision (91266)  
**Task**: [Kaggle Whale Identification Playground](https://www.kaggle.com/competitions/whale-categorization-playground)  
**Authors**: Kristoffer Osen & Sebastian Munthe


## Setup

In [ ]:
import sys
import os
import torch
import json
from dataset import build_retrieval_eval_loaders
from dataset import get_eval_config_and_data
from visualization import plot_embedding_tsne


sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))

from config import ExperimentConfig
from experiment import ExperimentManager
from dataset import (
    prepare_data,
    build_dataloaders,
 )
from models import build_model
from training import train
from evaluation import evaluate_retrieval
from visualization import plot_training_history

device = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if hasattr(torch.backends, "mps") and torch.backends.mps.is_available()
                      else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Contrastive Loss Training

**Contrastive Loss** operates on **pairs** rather
than triplets. For each pair of embeddings, the loss pulls same-class pairs together and
pushes different-class pairs apart:

$$L = (1 - y) \cdot \tfrac{1}{2} \max\{0,\ D - m^+\}^2 \;+\; y \cdot \tfrac{1}{2} \max\{0,\ m^- - D\}^2$$

where $D = \|f(x_i) - f(x_j)\|$ is the Euclidean distance, $y = 0$ for positive pairs
(same class) and $y = 1$ for negative pairs (different class).

Key implementation details:
- **Positive margin $m^+$**: without it, the model is incentivized to collapse all
  same-class embeddings to a single point. The positive margin creates a "good enough"
  threshold — once a positive pair is closer than $m^+$, the gradient stops and the model
  can spend capacity on harder problems.
- **Dead zone**: because $m^+ < m^-$, there is a range of distances where neither branch
  produces gradient, which stabilizes training.
- **PK Sampling**: same rationale as triplet — each batch needs P identities × K images
  to guarantee positive pairs exist. Without it, random batches may contain mostly
  singletons and produce almost no positive-pair signal.
- **Same limitation as triplet**: classes with only 1 image cannot form positive pairs
  and are effectively skipped during training.

The model uses `head_type="none"` — no classifier, only the L2-normalized embedding.
Validation uses retrieval recall (nearest-neighbor in embedding space) rather than
classification accuracy.

In [ ]:
config = ExperimentConfig(
    # ── Metadata ─────────────────────────────────────────────────
    experiment_name="contrastive_effnetb5",
    description="Contrastive loss with positive margin, PK sampling",

    # ── Paths ────────────────────────────────────────────────────
    data_dir="data",
    experiments_root="experiments",

    # ── Model ────────────────────────────────────────────────────
    backbone="efficientnet_b5",
    image_size=(456, 456),
    embedding_dim=512,
    head_type="none",          # No classifier, embedding-only

    # ── Training ─────────────────────────────────────────────────
    freeze_backbone_epochs=2,
    freeze_bn=True,
    epochs=20,
    batch_size=8,              
    accumulation_steps=2,     
    use_amp=True,

    # ── PK Sampling ──────────────────────────────────────────────
    pk_sampling=True,
    pk_p=4,                   
    pk_k=4,                    
    pk_min_samples=2,         

    # ── Loss ─────────────────────────────────────────────────────
    loss_type="contrastive",
    contrastive_pos_margin=0.2,
    contrastive_neg_margin=1.0,
    use_class_weights=False,  

    # ── Optimizer ────────────────────────────────────────────────
    lr_head=5e-4,
    lr_backbone=5e-5,
    weight_decay=1e-4,

    # ── Scheduler ────────────────────────────────────────────────
    scheduler="cosine",
    warmup_epochs=1,

    # ── Early stopping ───────────────────────────────────────────
    early_stopping_patience=6,
    early_stopping_metric="retrieval_recall@1",

    # ── Initialize from baseline ─────────────────────────────────
    init_from_checkpoint=BASELINE_CHECKPOINT if 'BASELINE_CHECKPOINT' in dir() else None,

    seed=42,
)

print("Contrastive config:")
print(config.summary())


## Data Preparation

In [ ]:
data = prepare_data(config)

print(f"\nClass mapping: {data['num_classes']} known whale identities")
print(f"ID → Index examples: {dict(list(data['id_to_idx'].items())[:5])} ...")

train_loader, val_loader = build_dataloaders(config, data)

## Train

In [ ]:
TRAINING_ENABLED = True
EXPERIMENT_ID = "unknown"

if TRAINING_ENABLED:
    # manager = train(config, data, device, resume_from=EXPERIMENT_ID)
    manager = train(config, data, device)
else:
    print("Skipping training")
    config = ExperimentConfig.load("experiments/" + EXPERIMENT_ID + "/config.json")
    manager = ExperimentManager(config, EXPERIMENT_ID)

## Training analysis

In [ ]:
metrics_path = manager.root / "metrics.json"

with open(metrics_path, "r") as f:
    metrics = json.load(f)

plot_training_history(metrics, title=config.experiment_name)

In [ ]:
print("\nFinal metrics (last epoch):")
for key in ["retrieval_recall@1", "retrieval_recall@5", "train_loss"]:
    if key in metrics:
        val = metrics[key][-1]
        if "acc" in key or "detection" in key or "recall" in key:
            print(f"  {key:<30s}: {val * 100:.2f}%")
        else:
            print(f"  {key:<30s}: {val:.4f}")


### Embedding quality

In [ ]:
eval_config, data = get_eval_config_and_data()

triplet_model = build_model(config, data["num_classes"], device)
manager.load_checkpoint(triplet_model, checkpoint="best")

train_loader, val_loader = build_retrieval_eval_loaders(eval_config, data)

print("Triplet retrieval performance:")
triplet_retrieval = evaluate_retrieval(
    triplet_model, train_loader, val_loader, eval_config, device,
)

plot_embedding_tsne(triplet_model, val_loader, data, device,
                    title="Triplet Loss Embedding — t-SNE", combine_loaders=[train_loader])